# Plant Disease Detection

## Group Members
1. Ahmad Saeed
2. Rehan Ahmad
3. Muaaz Anwar

## About Project
Our project is about detecting plant diseases using leaf images. We are using deep learning to predict plant diseases.

## Dataset Details
- Dataset Name: PlantVillage
- Dataset Source: Kaggle
- Total Images: 20,638 (approximately)
- Total Classes: 15
- Image Size: 224 × 224

## Model
We are using MobileNetV2 with TensorFlow and Keras.

## Results
- Training Accuracy: 89.71%
- Validation Accuracy: 90.87%

We trained our model for 5 epochs. These are our initial validation results.

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if len(files) > 0:
        print(files[:5])

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"

classes = os.listdir(DATASET_PATH)

print("Total classes:", len(classes))
print(classes)

In [ ]:
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(15, activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=1,
    min_lr=0.00001
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Number of classes:", len(class_names))
print(class_names)

In [ ]:
import tensorflow as tf

DATASET_PATH = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Number of classes:", len(class_names))
print(class_names)

In [ ]:
print(model)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation="softmax")
])

# Model ko explicitly build karna
model.build((None, 224, 224, 3))

print(model)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=1,
    min_lr=0.00001
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
model.save("/kaggle/working/plant_disease_mobilenetv2_best.keras")

print("Backup model saved successfully!")

In [ ]:
base_model.trainable = True

# Sirf last 30 layers train honge
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning setup complete!")

In [ ]:
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=[
        EarlyStopping(
            monitor="val_accuracy",
            patience=1,
            restore_best_weights=True
        )
    ]
)

In [ ]:
model.save("/kaggle/working/plant_disease_mobilenetv2_final.keras")

print("Final model saved successfully!")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
print("val_ds exists:", "val_ds" in globals())
print("DATASET_PATH exists:", "DATASET_PATH" in globals())
print("IMG_SIZE exists:", "IMG_SIZE" in globals())
print("BATCH_SIZE exists:", "BATCH_SIZE" in globals())
print("SEED exists:", "SEED" in globals())

In [ ]:

print("DATASET_PATH:", DATASET_PATH)

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("SEED:", SEED)

print("Dataset folders:")
import os
print(os.listdir(DATASET_PATH)[:10])


In [ ]:

import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("Training dataset created successfully")
print("Validation dataset created successfully")


In [ ]:

print("Model exists:", "model" in globals())

if "model" in globals():
    model.summary()


In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")


In [ ]:

print("TensorFlow:", "tf" in globals())
print("train_ds:", "train_ds" in globals())
print("val_ds:", "val_ds" in globals())
print("model:", "model" in globals())


In [ ]:

import os
import tensorflow as tf

DATASET_PATH = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("TensorFlow version:", tf.__version__)
print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Number of classes:", len(os.listdir(DATASET_PATH)))


In [ ]:

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Training and validation datasets restored!")


In [ ]:

from tensorflow.keras import layers, models

NUM_CLASSES = len(class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model created and compiled successfully!")
print("Number of classes:", NUM_CLASSES)


In [ ]:

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


In [ ]:

model.save("/kaggle/working/plant_disease_model.keras")

print("Model saved successfully!")


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

y_true = np.concatenate([
    labels.numpy() for images, labels in val_ds
])

y_pred = np.concatenate([
    np.argmax(model.predict(images, verbose=0), axis=1)
    for images, labels in val_ds
])

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Plant Disease Detection - Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Correct evaluation accuracy:", accuracy_score(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    labels=range(len(class_names)),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))

plt.figure(figsize=(16, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Plant Disease Detection - Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

import os
import matplotlib.pyplot as plt

class_counts = {
    class_name: len(os.listdir(os.path.join(DATASET_PATH, class_name)))
    for class_name in sorted(os.listdir(DATASET_PATH))
    if os.path.isdir(os.path.join(DATASET_PATH, class_name))
}

plt.figure(figsize=(14, 7))
plt.bar(class_counts.keys(), class_counts.values())
plt.title("PlantVillage Dataset - Class Distribution")
plt.xlabel("Plant Disease Classes")
plt.ylabel("Number of Images")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print("Total images:", sum(class_counts.values()))
print("Total classes:", len(class_counts))


In [ ]:

model.save("/kaggle/working/plant_disease_model.keras")
print("Model saved successfully!")


In [ ]:

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

image_paths = []

for class_name in class_names:
    class_path = os.path.join(DATASET_PATH, class_name)

    for file_name in os.listdir(class_path):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(
                (os.path.join(class_path, file_name), class_name)
            )

image_path, actual_class = random.choice(image_paths)

img = tf.keras.utils.load_img(
    image_path,
    target_size=IMG_SIZE
)

img_array = tf.keras.utils.img_to_array(img)
prediction = model.predict(
    tf.expand_dims(img_array, axis=0),
    verbose=0
)[0]

predicted_index = np.argmax(prediction)
predicted_class = class_names[predicted_index]
confidence = prediction[predicted_index] * 100

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(
    f"Predicted: {predicted_class}\n"
    f"Confidence: {confidence:.2f}%"
)
plt.show()

print("Actual class:", actual_class)
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.2f}%")


In [ ]:

model.save("/kaggle/working/plant_disease_model.keras")
print("Model saved successfully!")
